# 🧠 Tripolar EEG Decomposition Notebook — SK1
**Subject:** SK1 &nbsp;|&nbsp; **Date:** February 19, 2026 &nbsp;|&nbsp; **Recorded:** 10:58:47 AM  
**Workspace:** `BAmp_Felt_VEP_11ch_500ms-wnotch`

---

## Experiment Overview

### Hardware
- **Amplifier:** BrainAmp (Brain Products)
- **Reference electrode:** Behind the ear (mastoid) — dedicated amplifier input, **not** one of the 11 data channels
- **Ground electrode:** Base of skull — dedicated amplifier input
- **Recording:** 11 channels at **1000 Hz**, 16-bit (INT16), resolution **0.1 µV/bit**
- **Hardware filters:** 0.1 Hz high-pass (10s time constant), 250 Hz low-pass, notch OFF during recording

### Electrode Setup
Each **tripolar concentric ring electrode (CRE)** outputs two signals:
- **Conventional derivation (Conv):** outer ring referenced to the mastoid — behaves like a standard EEG electrode
- **tEEG derivation (Laplacian):** computed on-board from the 3 concentric rings — surface Laplacian, spatially filtered

| Channel | Electrode | Derivation | Electrolyte |
|---------|-----------|------------|-------------|
| Ch1 | Tripolar CRE #1 | Conventional | Saltwater |
| Ch2 | Tripolar CRE #1 | tEEG (Laplacian) | Saltwater |
| Ch3 | Tripolar CRE #2 | Conventional | Saltwater |
| Ch4 | Tripolar CRE #2 | tEEG (Laplacian) | Saltwater |
| Ch5 | Tripolar CRE #3 | Conventional | Saltwater |
| Ch6 | Tripolar CRE #3 | tEEG (Laplacian) | Saltwater |
| Ch7 | Tripolar CRE #4 | Conventional | Saltwater |
| Ch8 | Tripolar CRE #4 | tEEG (Laplacian) | Saltwater |
| Ch9 | Tripolar CRE #5 | Conventional | Conductive paste |
| Ch10 | Tripolar CRE #5 | tEEG (Laplacian) | Conductive paste |
| Ch11 | Standard disc electrode | Conventional | Conductive paste |
| (Ref) | Mastoid (behind ear) | — | — |
| (Gnd) | Base of skull | — | — |

> ⚠️ **Please verify** the odd=Conv / even=tEEG pairing and the electrode positions with your professor.

### Paradigm
1. **Checkerboard VEP** — 3 blocks of 20 reversals (S7 triggers, ~600ms ISI) → visually evoked potentials + induced alpha
2. **Eyes open / close** — alternating trials (~30s each) → spontaneous alpha (Berger effect)

### Goal
Determine if **tripolar tEEG electrodes** detect visually-induced alpha waves and spontaneous alpha as effectively as **conventional disc electrodes**.

---

### Event Timeline (from .vmrk)

| Time | Event |
|------|-------|
| 50.5s – 62.0s | Stim Block 1 (20× S7 checkerboard triggers) |
| 95.3s – 106.9s | Stim Block 2 (20× S7) |
| 138.5s – 149.8s | Stim Block 3 (20× S7) |
| 164.1s | Eyes Open #1 |
| 192.8s | Eyes Close #1 |
| 222.3s | Eyes Open #2 |
| 252.5s | Eyes Close #2 |
| 282.2s | Eyes Open #3 |
| 313.1s | Eyes Close #3 |
| 342.5s | Eyes Open #4 |


## 1. Setup & Data Loading

In [ ]:
# ─── Dependencies ───
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.signal import butter, filtfilt, hilbert
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
print("Libraries loaded ✓")

In [ ]:
# ═══════════════════════════════════════
# CONFIGURATION — Update paths as needed
# ═══════════════════════════════════════
EEG_FILE = 'SK1_2-19-2026.eeg'
AVG_FILE = 'SK1_2-19-2026-Triggers.avg'

# From .vhdr
N_CHANNELS = 11
FS = 1000             # Hz
RESOLUTION = 0.1      # µV per bit

# Channel labels — verify with your professor!
CH_LABELS = [
    'Ch1 – SW CRE #1 (Conv)',
    'Ch2 – SW CRE #1 (tEEG)',
    'Ch3 – SW CRE #2 (Conv)',
    'Ch4 – SW CRE #2 (tEEG)',
    'Ch5 – SW CRE #3 (Conv)',
    'Ch6 – SW CRE #3 (tEEG)',
    'Ch7 – SW CRE #4 (Conv)',
    'Ch8 – SW CRE #4 (tEEG)',
    'Ch9 – Paste CRE #5 (Conv)',
    'Ch10 – Paste CRE #5 (tEEG)',
    'Ch11 – Standard Disc (Conv)',
]
SHORT = [f'Ch{i+1}' for i in range(N_CHANNELS)]

# Electrode type grouping
IDX_SW_CONV  = [0, 2, 4, 6]     # Saltwater conventional
IDX_SW_TEEG  = [1, 3, 5, 7]     # Saltwater tEEG
IDX_PASTE_CONV = [8]             # Paste conventional
IDX_PASTE_TEEG = [9]             # Paste tEEG
IDX_DISC = [10]                  # Standard disc

print("Configuration set ✓")

In [ ]:
# ─── Load raw EEG ───
raw = np.fromfile(EEG_FILE, dtype=np.int16)
n_samples = len(raw) // N_CHANNELS
assert len(raw) % N_CHANNELS == 0, "File not evenly divisible by 11!"

# Reshape (multiplexed) and scale to µV
eeg = raw[:n_samples * N_CHANNELS].reshape(n_samples, N_CHANNELS).T.astype(np.float64)
eeg *= RESOLUTION  # → µV

t = np.arange(n_samples) / FS

print(f"EEG shape:  {eeg.shape}  (channels × samples)")
print(f"Duration:   {n_samples/FS:.1f} s  ({n_samples/FS/60:.1f} min)")
print(f"Units:      µV")
print(f"Reference:  Mastoid (behind ear, dedicated amp input)")
print(f"Ground:     Base of skull (dedicated amp input)")

In [ ]:
# ─── Parse event markers (from .vmrk) ───

# Stimulus triggers (S7) — checkerboard pattern reversals
stim_samples = np.array([
    # Block 1
    50518, 51093, 51691, 52270, 52861, 53460, 54058, 54687, 55283, 55868,
    56467, 57064, 57718, 58294, 58896, 59503, 60117, 60757, 61384, 61993,
    # Block 2
    95295, 95904, 96520, 97123, 97731, 98317, 98907, 99557, 100172, 100772,
    101391, 101993, 102628, 103225, 103831, 104440, 105048, 105673, 106282, 106889,
    # Block 3
    138451, 139038, 139654, 140272, 140864, 141483, 142060, 142655, 143234, 143840,
    144431, 145022, 145649, 146239, 146847, 147472, 148063, 148690, 149280, 149848,
])
stim_times = stim_samples / FS

# Block boundaries (for shading on plots)
stim_blocks = [(50.5, 62.0), (95.3, 107.0), (138.5, 150.0)]

# Eyes open/close markers
events_oc = [
    ('open',  164141), ('close', 192761), ('open',  222341),
    ('close', 252481), ('open',  282221), ('close', 313061), ('open',  342521),
]

# Build epoch list (each epoch runs from one marker to the next)
epochs_oc = []
for i in range(len(events_oc) - 1):
    lbl, start = events_oc[i]
    _, end = events_oc[i + 1]
    epochs_oc.append({'label': lbl, 'start': start, 'end': end,
                      'start_s': start/FS, 'end_s': end/FS, 'dur': (end-start)/FS})
# Last epoch: open until +30s or end of recording
last_start = events_oc[-1][1]
last_end = min(last_start + 30*FS, n_samples)
epochs_oc.append({'label': 'open', 'start': last_start, 'end': last_end,
                  'start_s': last_start/FS, 'end_s': last_end/FS, 'dur': (last_end-last_start)/FS})

open_epochs  = [ep for ep in epochs_oc if ep['label'] == 'open']
close_epochs = [ep for ep in epochs_oc if ep['label'] == 'close']

print(f"Stimulus triggers: {len(stim_samples)} events in 3 blocks of 20")
print(f"\nEyes Open/Close Epochs:")
for ep in epochs_oc:
    print(f"  {ep['label']:>5s}: {ep['start_s']:6.1f}s – {ep['end_s']:6.1f}s  ({ep['dur']:.1f}s)")

In [ ]:
# ─── Load pre-averaged ERP (from BrainVision Recorder) ───
# -Triggers.avg: 500 pts × 11 ch × float32
# Pre-filtered: 0.5 Hz HP, 30 Hz LP, 60 Hz notch, baseline-corrected
# 100ms pre-stimulus, 400ms post-stimulus, averaged across 74 segments

avg_raw = np.fromfile(AVG_FILE, dtype=np.float32)
AVG_PTS = 500
avg_data = avg_raw[:AVG_PTS * N_CHANNELS].reshape(AVG_PTS, N_CHANNELS).T  # (11, 500) µV
avg_t = np.arange(AVG_PTS) / FS * 1000 - 100  # ms, t=0 at stimulus

print(f"Averaged ERP:   {avg_data.shape} (channels × timepoints)")
print(f"Time window:    {avg_t[0]:.0f} to {avg_t[-1]:.0f} ms")
print(f"Segments avg'd: 74")
print(f"Filters:        0.5–30 Hz, 60 Hz notch, baseline-corrected")

## 2. Helper Functions

In [ ]:
def notch_filter(data, freq=60, Q=30, fs=FS):
    """Remove power line noise."""
    b, a = signal.iirnotch(freq, Q, fs)
    return filtfilt(b, a, data)

def bandpass_filter(data, low, high, fs=FS, order=4):
    """Butterworth bandpass filter."""
    b, a = butter(order, [low/(fs/2), high/(fs/2)], btype='band')
    return filtfilt(b, a, data)

def compute_envelope(data, smooth_s=1.0, fs=FS):
    """Amplitude envelope via Hilbert transform + smoothing."""
    analytic = hilbert(data)
    env = np.abs(analytic)
    kernel = np.ones(int(fs * smooth_s)) / int(fs * smooth_s)
    return np.convolve(env, kernel, mode='same')

# EEG frequency bands
BANDS = {'Delta': (1,4), 'Theta': (4,8), 'Alpha': (8,13), 'Beta': (13,30), 'Gamma': (30,45)}
BAND_COLORS = {'Delta':'#2c3e50', 'Theta':'#8e44ad', 'Alpha':'#e67e22',
               'Beta':'#27ae60', 'Gamma':'#c0392b'}

# Color scheme by electrode type
def ch_color(i):
    if i in IDX_SW_TEEG:    return '#3498db'   # Saltwater tEEG (blue)
    if i in IDX_PASTE_TEEG: return '#2ecc71'   # Paste tEEG (green)
    if i in IDX_DISC:       return '#e74c3c'   # Conv disc (red)
    if i in IDX_PASTE_CONV: return '#9b59b6'   # Paste conv (purple)
    return '#95a5a6'                            # Saltwater conv (gray)

def add_event_markers(ax, shade_blocks=True, shade_close=True):
    """Add experimental event markers to any axis."""
    if shade_blocks:
        for bs, be in stim_blocks:
            ax.axvspan(bs, be, alpha=0.10, color='red')
    for lbl, samp in events_oc:
        c = '#27ae60' if lbl == 'open' else '#3498db'
        ax.axvline(samp/FS, color=c, linewidth=0.7, alpha=0.6, linestyle='--')
    if shade_close:
        for ep in close_epochs:
            ax.axvspan(ep['start_s'], ep['end_s'], alpha=0.06, color='#3498db')

# Legend items for event markers
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
EVENT_LEGEND = [
    Patch(facecolor='red', alpha=0.15, label='Stim blocks (S7)'),
    Patch(facecolor='#3498db', alpha=0.12, label='Eyes closed'),
    Line2D([0],[0], color='#27ae60', linestyle='--', label='Eyes open'),
    Line2D([0],[0], color='#3498db', linestyle='--', label='Eyes close'),
]

ELEC_LEGEND = [
    Patch(facecolor='#95a5a6', label='SW Conv (Ch1,3,5,7)'),
    Patch(facecolor='#3498db', label='SW tEEG (Ch2,4,6,8)'),
    Patch(facecolor='#9b59b6', label='Paste Conv (Ch9)'),
    Patch(facecolor='#2ecc71', label='Paste tEEG (Ch10)'),
    Patch(facecolor='#e74c3c', label='Standard Disc (Ch11)'),
]

print("Helpers defined ✓")

## 3. Channel Statistics

In [ ]:
print(f"{'Channel':<30} {'Min(µV)':>9} {'Max(µV)':>9} {'Std(µV)':>9} {'Clipping?':>10}")
print("═" * 72)
for i in range(N_CHANNELS):
    clips = "⚠ YES" if eeg[i].max() >= 3276.6 or eeg[i].min() <= -3276.6 else ""
    print(f"{CH_LABELS[i]:<30} {eeg[i].min():>9.1f} {eeg[i].max():>9.1f} "
          f"{eeg[i].std():>9.1f} {clips:>10}")

print()
print("Note: Clipping channels hit ±3276.7 µV (int16 × 0.1 µV/bit saturation)")
print("      tEEG channels have much lower amplitude — this is expected because")
print("      the Laplacian derivation cancels far-field/volume-conducted signals.")

## 4. Raw Time Series with Event Markers

Full recording with stimulus blocks (red shading), eyes-open (green dashes), eyes-closed (blue dashes/shading).

In [ ]:
fig, axes = plt.subplots(11, 1, figsize=(18, 22), sharex=True)
fig.suptitle('Raw EEG (µV) — Full Recording with Event Markers', fontsize=14, fontweight='bold', y=1.0)

ds = 10
for i in range(N_CHANNELS):
    ax = axes[i]
    ax.plot(t[::ds], eeg[i, ::ds], linewidth=0.3, color='#2c3e50')
    ax.set_ylabel(f'Ch{i+1}', fontsize=9)
    ymax = np.percentile(np.abs(eeg[i]), 99.5)
    ax.set_ylim(-ymax, ymax)
    ax.tick_params(labelsize=8)
    add_event_markers(ax)

axes[0].legend(handles=EVENT_LEGEND, fontsize=7, loc='upper right', ncol=4)
axes[0].set_xlim(0, t[-1])
axes[-1].set_xlabel('Time (s)', fontsize=11)
plt.tight_layout(); plt.show()

## 5. Power Spectral Density

### 5a. Full range (0–80 Hz) — identifies 60 Hz noise
### 5b. After 60 Hz notch, zoomed to 1–30 Hz — focuses on alpha

In [ ]:
# 5a: Full range PSD
fig, axes = plt.subplots(6, 2, figsize=(16, 20))
fig.suptitle('PSD (Welch, 4096-pt) — 0 to 80 Hz', fontsize=14, fontweight='bold', y=1.0)

for i in range(N_CHANNELS):
    ax = axes.flatten()[i]
    f, pxx = signal.welch(eeg[i], fs=FS, nperseg=4096)
    mask = f <= 80
    ax.semilogy(f[mask], pxx[mask], linewidth=1.2, color='#e74c3c')
    amask = (f >= 8) & (f <= 13) & mask
    ax.fill_between(f[amask], pxx[amask], alpha=0.3, color='#3498db', label='Alpha 8–13 Hz')
    ax.set_title(CH_LABELS[i], fontsize=9, fontweight='bold')
    ax.set_xlabel('Hz', fontsize=8); ax.set_ylabel('µV²/Hz', fontsize=8)
    ax.legend(fontsize=7); ax.tick_params(labelsize=7)
    ax.axvline(60, color='gray', ls='--', alpha=0.5, lw=0.8)
    ax.set_xlim(0, 80)
axes.flatten()[11].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# 5b: Notch-filtered, zoomed to 1–30 Hz
fig, axes = plt.subplots(6, 2, figsize=(16, 20))
fig.suptitle('PSD After 60 Hz Notch — 1 to 30 Hz (Alpha Highlighted)', fontsize=14, fontweight='bold', y=1.0)

for i in range(N_CHANNELS):
    ax = axes.flatten()[i]
    cleaned = notch_filter(eeg[i])
    f, pxx = signal.welch(cleaned, fs=FS, nperseg=4096)
    mask = (f >= 1) & (f <= 30)
    ax.plot(f[mask], pxx[mask], linewidth=1.5, color='#2c3e50')
    amask = (f >= 8) & (f <= 13) & mask
    ax.fill_between(f[amask], pxx[amask], alpha=0.4, color='#e67e22', label='Alpha 8–13 Hz')
    ax.set_title(CH_LABELS[i], fontsize=9, fontweight='bold')
    ax.set_xlabel('Hz', fontsize=8); ax.set_ylabel('µV²/Hz', fontsize=8)
    ax.legend(fontsize=7); ax.tick_params(labelsize=7)
axes.flatten()[11].set_visible(False)
plt.tight_layout(); plt.show()

## 6. Time-Frequency Spectrograms — Per Channel

**Top:** bandpass-filtered (1–45 Hz) trace. **Bottom:** spectrogram (0–45 Hz). Cyan lines = alpha band (8–13 Hz). Event markers overlaid. Look for bright horizontal bands in the alpha range during eyes-closed periods.

In [ ]:
for i in range(N_CHANNELS):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 7),
                                    gridspec_kw={'height_ratios': [1, 3]})
    fig.suptitle(f'Time-Frequency — {CH_LABELS[i]}', fontsize=12, fontweight='bold')
    
    cleaned = notch_filter(eeg[i])
    
    # Top: bandpass trace
    bp = bandpass_filter(cleaned, 1, 45)
    ax1.plot(t[::10], bp[::10], linewidth=0.3, color='#34495e')
    ax1.set_ylabel('µV'); ax1.set_xlim(0, t[-1])
    ax1.set_title('Bandpass 1–45 Hz', fontsize=10)
    add_event_markers(ax1)
    
    # Bottom: spectrogram
    nperseg = 2048
    f_s, t_s, Sxx = signal.spectrogram(cleaned, fs=FS, nperseg=nperseg,
                                        noverlap=nperseg//2, nfft=4096)
    fm = f_s <= 45
    im = ax2.pcolormesh(t_s, f_s[fm], 10*np.log10(Sxx[fm]+1e-10),
                         shading='gouraud', cmap='inferno', vmin=-20)
    ax2.set_ylabel('Frequency (Hz)'); ax2.set_xlabel('Time (s)')
    ax2.set_ylim(1, 45)
    ax2.axhline(8, color='cyan', lw=0.8, ls='--', alpha=0.7)
    ax2.axhline(13, color='cyan', lw=0.8, ls='--', alpha=0.7)
    ax2.text(5, 10.5, 'α', color='cyan', fontsize=10, fontweight='bold')
    add_event_markers(ax2, shade_close=False)
    plt.colorbar(im, ax=ax2, label='dB')
    
    plt.tight_layout(); plt.show()
    print()

## 7. Band Decomposition — Per Channel

Each channel split into 5 bands: **Delta** (1–4), **Theta** (4–8), **Alpha** (8–13) ← target, **Beta** (13–30), **Gamma** (30–45 Hz). Event markers overlaid.

In [ ]:
for i in range(N_CHANNELS):
    fig, axs = plt.subplots(len(BANDS)+1, 1, figsize=(16, 10), sharex=True,
                             gridspec_kw={'height_ratios': [2]+[1]*len(BANDS)})
    fig.suptitle(f'Band Decomposition — {CH_LABELS[i]}', fontsize=12, fontweight='bold')
    
    cleaned = notch_filter(eeg[i])
    ds = 20
    
    bp_full = bandpass_filter(cleaned, 1, 45)
    axs[0].plot(t[::ds], bp_full[::ds], lw=0.3, color='#2c3e50')
    axs[0].set_title('Broadband (1–45 Hz)', fontsize=10); axs[0].set_ylabel('µV', fontsize=8)
    
    for j, (bname, (lo, hi)) in enumerate(BANDS.items()):
        ax = axs[j+1]
        bp = bandpass_filter(cleaned, lo, hi)
        ax.plot(t[::ds], bp[::ds], lw=0.4, color=BAND_COLORS[bname])
        ax.set_title(f'{bname} ({lo}–{hi} Hz)', fontsize=10, color=BAND_COLORS[bname])
        ax.set_ylabel('µV', fontsize=8)
    
    for ax in axs:
        ax.set_xlim(0, t[-1]); ax.tick_params(labelsize=7)
        add_event_markers(ax)
    
    axs[-1].set_xlabel('Time (s)', fontsize=10)
    plt.tight_layout(); plt.show()
    print()

## 8. Alpha Envelope Over Time

Instantaneous alpha (8–13 Hz) power via Hilbert transform (1s smoothing). Eyes-closed periods are shaded blue — you should see alpha increase there (Berger effect).

In [ ]:
fig, axes = plt.subplots(11, 1, figsize=(17, 24), sharex=True)
fig.suptitle('Alpha Band (8–13 Hz) Envelope', fontsize=14, fontweight='bold', y=1.0)

alpha_envelopes = []
ds = 50

for i in range(N_CHANNELS):
    ax = axes[i]
    cleaned = notch_filter(eeg[i])
    alpha = bandpass_filter(cleaned, 8, 13)
    env = compute_envelope(alpha)
    alpha_envelopes.append(env)
    
    ax.plot(t[::ds], env[::ds], lw=1, color='#e74c3c')
    ax.fill_between(t[::ds], 0, env[::ds], alpha=0.2, color='#e74c3c')
    ax.set_ylabel(f'Ch{i+1}', fontsize=9)
    ax.tick_params(labelsize=8)
    add_event_markers(ax)

axes[0].legend(handles=EVENT_LEGEND, fontsize=7, loc='upper right', ncol=4)
axes[0].set_xlim(0, t[-1])
axes[-1].set_xlabel('Time (s)', fontsize=11)
plt.tight_layout(); plt.show()

## 9. Eyes Open vs Eyes Closed — Alpha Power Comparison

Core analysis for **spontaneous alpha**. Classic expectation: alpha power **increases** with eyes closed (Berger effect, 1929). We compute PSD separately for open vs closed epochs and compare.

In [ ]:
fig, axes = plt.subplots(6, 2, figsize=(16, 20))
fig.suptitle('PSD: Eyes Open (green) vs Eyes Closed (blue) — 1–30 Hz', fontsize=14, fontweight='bold', y=1.0)

alpha_open = []
alpha_closed = []

for i in range(N_CHANNELS):
    ax = axes.flatten()[i]
    cleaned = notch_filter(eeg[i])
    
    open_data  = np.concatenate([cleaned[ep['start']:ep['end']] for ep in open_epochs])
    close_data = np.concatenate([cleaned[ep['start']:ep['end']] for ep in close_epochs])
    
    f_o, pxx_o = signal.welch(open_data, fs=FS, nperseg=4096)
    f_c, pxx_c = signal.welch(close_data, fs=FS, nperseg=4096)
    
    mask = (f_o >= 1) & (f_o <= 30)
    ax.plot(f_o[mask], pxx_o[mask], lw=1.5, color='#27ae60', label='Eyes Open')
    ax.plot(f_c[mask], pxx_c[mask], lw=1.5, color='#3498db', label='Eyes Closed')
    
    amask = (f_o >= 8) & (f_o <= 13) & mask
    ax.fill_between(f_o[amask], pxx_c[amask], pxx_o[amask], alpha=0.2, color='#e74c3c')
    
    ax.set_title(CH_LABELS[i], fontsize=9, fontweight='bold')
    ax.set_xlabel('Hz', fontsize=8); ax.set_ylabel('µV²/Hz', fontsize=8)
    ax.legend(fontsize=7); ax.tick_params(labelsize=7)
    
    alpha_open.append(np.mean(pxx_o[(f_o >= 8) & (f_o <= 13)]))
    alpha_closed.append(np.mean(pxx_c[(f_c >= 8) & (f_c <= 13)]))

axes.flatten()[11].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# ─── Alpha reactivity: bar chart ───
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Alpha Power (8–13 Hz): Eyes Open vs Eyes Closed', fontsize=13, fontweight='bold')

x = np.arange(N_CHANNELS)
w = 0.35

ax1.bar(x-w/2, alpha_open, w, color='#27ae60', label='Eyes Open', edgecolor='k', lw=0.5)
ax1.bar(x+w/2, alpha_closed, w, color='#3498db', label='Eyes Closed', edgecolor='k', lw=0.5)
ax1.set_xticks(x); ax1.set_xticklabels(SHORT, fontsize=9)
ax1.set_ylabel('Alpha Power (µV²/Hz)'); ax1.set_title('Absolute'); ax1.legend(); ax1.set_yscale('log')

ratio = np.array(alpha_closed) / (np.array(alpha_open) + 1e-10)
ax2.bar(x, ratio, color=[ch_color(i) for i in range(N_CHANNELS)], edgecolor='k', lw=0.5)
ax2.axhline(1, color='gray', lw=1, ls='--')
ax2.set_xticks(x); ax2.set_xticklabels(SHORT, fontsize=9)
ax2.set_ylabel('Ratio (Closed / Open)')
ax2.set_title('Alpha Reactivity (>1 = Berger effect detected)')
ax2.legend(handles=ELEC_LEGEND, fontsize=7, loc='upper right')

plt.tight_layout(); plt.show()

print("\nAlpha Reactivity (Eyes Closed / Eyes Open):")
for i in range(N_CHANNELS):
    arrow = "↑ Berger+" if ratio[i] > 1 else "↓"
    print(f"  {SHORT[i]:>4s}: {ratio[i]:5.2f}x  {arrow}")

## 10. Visual Evoked Potential (VEP)

Pre-averaged by BrainVision Recorder: 74 checkerboard triggers, -100 to +400 ms, filtered 0.5–30 Hz + 60 Hz notch, baseline-corrected.

In [ ]:
# 10a: Individual channel ERPs
fig, axes = plt.subplots(6, 2, figsize=(16, 18))
fig.suptitle('VEP — Averaged Across 74 Stimulus Triggers (per channel)', fontsize=14, fontweight='bold', y=1.0)

for i in range(N_CHANNELS):
    ax = axes.flatten()[i]
    ax.plot(avg_t, avg_data[i], lw=1.5, color=ch_color(i))
    ax.axvline(0, color='red', lw=1, ls='--', alpha=0.7, label='Stimulus')
    ax.axhline(0, color='gray', lw=0.5)
    ax.fill_between(avg_t, avg_data[i], 0, alpha=0.1, color=ch_color(i))
    ax.set_title(CH_LABELS[i], fontsize=9, fontweight='bold')
    ax.set_xlabel('ms', fontsize=8); ax.set_ylabel('µV', fontsize=8)
    ax.set_xlim(-100, 400); ax.tick_params(labelsize=7)
    if i == 0: ax.legend(fontsize=7)

axes.flatten()[11].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# 10b: Overlay comparison by electrode type
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('VEP Comparison by Electrode Type', fontsize=13, fontweight='bold')

# Saltwater tEEG
ax = axes[0]
for idx in IDX_SW_TEEG:
    ax.plot(avg_t, avg_data[idx], lw=1.2, alpha=0.7, label=f'Ch{idx+1}')
ax.axvline(0, color='red', lw=1, ls='--', alpha=0.5); ax.axhline(0, color='gray', lw=0.3)
ax.set_title('Saltwater tEEG'); ax.set_xlabel('ms'); ax.set_ylabel('µV')
ax.legend(fontsize=8); ax.set_xlim(-100, 400)

# Paste tEEG vs Disc
ax = axes[1]
ax.plot(avg_t, avg_data[9], lw=2, color='#2ecc71', label='Ch10 Paste tEEG')
ax.plot(avg_t, avg_data[10], lw=2, color='#e74c3c', label='Ch11 Disc')
ax.plot(avg_t, avg_data[8], lw=1.5, color='#9b59b6', ls='--', label='Ch9 Paste Conv')
ax.axvline(0, color='red', lw=1, ls='--', alpha=0.5); ax.axhline(0, color='gray', lw=0.3)
ax.set_title('Paste tEEG vs Standard Disc'); ax.set_xlabel('ms')
ax.legend(fontsize=8); ax.set_xlim(-100, 400)

# Grand average by type
ax = axes[2]
avg_sw = np.mean(avg_data[IDX_SW_TEEG], axis=0)
ax.plot(avg_t, avg_sw, lw=2, color='#3498db', label='Avg SW tEEG')
ax.plot(avg_t, avg_data[9], lw=2, color='#2ecc71', label='Paste tEEG')
ax.plot(avg_t, avg_data[10], lw=2, color='#e74c3c', label='Disc')
ax.axvline(0, color='red', lw=1, ls='--', alpha=0.5); ax.axhline(0, color='gray', lw=0.3)
ax.set_title('Grand Average by Type'); ax.set_xlabel('ms')
ax.legend(fontsize=8); ax.set_xlim(-100, 400)

plt.tight_layout(); plt.show()

## 11. tEEG vs Conventional — Alpha Dynamics Comparison

Normalized alpha envelopes side-by-side. If tEEG electrodes are "as good as" conventional, the curves should track the same temporal pattern of alpha modulation.

In [ ]:
alpha_norm = [env / (np.percentile(env, 95) + 1e-10) for env in alpha_envelopes]

fig, axes = plt.subplots(3, 1, figsize=(17, 11), sharex=True)
fig.suptitle('Normalized Alpha Envelope — tEEG vs Conventional', fontsize=13, fontweight='bold', y=1.0)
ds = 50

# tEEG
ax = axes[0]
for idx in IDX_SW_TEEG:
    ax.plot(t[::ds], alpha_norm[idx][::ds], lw=0.8, alpha=0.7, label=f'Ch{idx+1}')
ax.plot(t[::ds], alpha_norm[9][::ds], lw=1.5, color='#e74c3c', label='Ch10 Paste tEEG')
ax.set_title('tEEG Channels'); ax.legend(fontsize=8, ncol=5, loc='upper right')
ax.set_ylabel('Norm. Power')

# Conventional
ax = axes[1]
for idx in IDX_SW_CONV:
    ax.plot(t[::ds], alpha_norm[idx][::ds], lw=0.8, alpha=0.7, label=f'Ch{idx+1}')
ax.plot(t[::ds], alpha_norm[8][::ds], lw=1.5, color='#e74c3c', label='Ch9 Paste Conv')
ax.plot(t[::ds], alpha_norm[10][::ds], lw=2, color='black', label='Ch11 Disc')
ax.set_title('Conventional Channels + Disc'); ax.legend(fontsize=8, ncol=6, loc='upper right')
ax.set_ylabel('Norm. Power')

# Grand
ax = axes[2]
avg_teeg = np.mean([alpha_norm[i] for i in IDX_SW_TEEG], axis=0)
ax.plot(t[::ds], avg_teeg[::ds], lw=2, color='#3498db', label='Avg SW tEEG')
ax.plot(t[::ds], alpha_norm[9][::ds], lw=2, color='#2ecc71', label='Paste tEEG')
ax.plot(t[::ds], alpha_norm[10][::ds], lw=2, color='#e74c3c', label='Disc')
ax.set_title('Grand Comparison'); ax.legend(fontsize=9, loc='upper right')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Norm. Power')

for ax in axes:
    ax.set_xlim(0, t[-1]); ax.tick_params(labelsize=8)
    add_event_markers(ax)

plt.tight_layout(); plt.show()

## 12. Cross-Channel Alpha Correlation

How well does the alpha envelope from each tEEG channel track the conventional disc (Ch11)? High correlation = the tEEG electrode captures the same neural events.

In [ ]:
# Compute correlation of each channel's alpha envelope with Ch11 (disc)
ref_env = alpha_envelopes[10]  # Ch11 disc
ds = 10
ref_ds = ref_env[::ds]

corrs = []
for i in range(N_CHANNELS):
    env_ds = alpha_envelopes[i][::ds]
    r = np.corrcoef(ref_ds, env_ds)[0, 1]
    corrs.append(r)

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(N_CHANNELS), corrs, color=[ch_color(i) for i in range(N_CHANNELS)],
       edgecolor='black', linewidth=0.5)
ax.set_xticks(range(N_CHANNELS))
ax.set_xticklabels(SHORT, fontsize=10)
ax.set_ylabel('Pearson r')
ax.set_title('Alpha Envelope Correlation with Ch11 (Standard Disc)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.axhline(1, color='gray', lw=0.5, ls='--')
ax.legend(handles=ELEC_LEGEND, fontsize=7, loc='upper left')

for i, r in enumerate(corrs):
    ax.text(i, r + 0.02, f'{r:.3f}', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout(); plt.show()

print("\nCorrelation with Ch11 (Disc):")
for i in range(N_CHANNELS):
    star = " ★" if corrs[i] > 0.8 else ""
    print(f"  {SHORT[i]:>4s}: r = {corrs[i]:.4f}{star}")

## 13. Summary Statistics

In [ ]:
# Alpha SNR
alpha_snr = []
for i in range(N_CHANNELS):
    cleaned = notch_filter(eeg[i])
    f, pxx = signal.welch(cleaned, fs=FS, nperseg=4096)
    ap  = np.mean(pxx[(f >= 8) & (f <= 13)])
    nap = np.mean(pxx[((f >= 4) & (f < 8)) | ((f > 13) & (f <= 30))])
    alpha_snr.append(10 * np.log10(ap / (nap + 1e-10)))

fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Summary Dashboard', fontsize=13, fontweight='bold')
colors = [ch_color(i) for i in range(N_CHANNELS)]

ratio = np.array(alpha_closed) / (np.array(alpha_open) + 1e-10)

ax1.bar(range(N_CHANNELS), alpha_snr, color=colors, edgecolor='k', lw=0.5)
ax1.set_xticks(range(N_CHANNELS)); ax1.set_xticklabels(SHORT, fontsize=7)
ax1.set_ylabel('dB'); ax1.set_title('Alpha SNR'); ax1.axhline(0, color='gray', lw=0.5)

ax2.bar(range(N_CHANNELS), ratio, color=colors, edgecolor='k', lw=0.5)
ax2.set_xticks(range(N_CHANNELS)); ax2.set_xticklabels(SHORT, fontsize=7)
ax2.set_ylabel('Closed/Open'); ax2.set_title('Alpha Reactivity'); ax2.axhline(1, color='gray', lw=0.5, ls='--')

ax3.bar(range(N_CHANNELS), corrs, color=colors, edgecolor='k', lw=0.5)
ax3.set_xticks(range(N_CHANNELS)); ax3.set_xticklabels(SHORT, fontsize=7)
ax3.set_ylabel('Pearson r'); ax3.set_title('Corr. with Disc (Ch11)')

# VEP peak-to-peak amplitude
vep_p2p = [avg_data[i].max() - avg_data[i].min() for i in range(N_CHANNELS)]
ax4.bar(range(N_CHANNELS), vep_p2p, color=colors, edgecolor='k', lw=0.5)
ax4.set_xticks(range(N_CHANNELS)); ax4.set_xticklabels(SHORT, fontsize=7)
ax4.set_ylabel('µV'); ax4.set_title('VEP Peak-to-Peak')

ax4.legend(handles=ELEC_LEGEND, fontsize=6, loc='upper right')
plt.tight_layout(); plt.show()

In [ ]:
# ─── Comprehensive table ───
print(f"{'Channel':<30} {'αSNR(dB)':>9} {'αReact':>8} {'CorrDisc':>9} {'VEP P2P':>9}")
print("═" * 70)
types = ['SW Conv','SW tEEG']*4 + ['Paste Conv','Paste tEEG','Disc']
for i in range(N_CHANNELS):
    print(f"{CH_LABELS[i]:<30} {alpha_snr[i]:>8.2f}  {ratio[i]:>7.2f}x {corrs[i]:>8.4f}  {vep_p2p[i]:>8.1f}")

print()
print("─── Group Averages ───")
groups = [
    ("SW tEEG (Ch2,4,6,8)",  IDX_SW_TEEG),
    ("SW Conv (Ch1,3,5,7)",   IDX_SW_CONV),
    ("Paste tEEG (Ch10)",     IDX_PASTE_TEEG),
    ("Paste Conv (Ch9)",      IDX_PASTE_CONV),
    ("Standard Disc (Ch11)",  IDX_DISC),
]
print(f"  {'Group':<26} {'αSNR':>8} {'αReact':>8} {'CorrDisc':>9} {'VEP P2P':>9}")
for name, idxs in groups:
    snr = np.mean([alpha_snr[i] for i in idxs])
    react = np.mean([ratio[i] for i in idxs])
    corr = np.mean([corrs[i] for i in idxs])
    vep = np.mean([vep_p2p[i] for i in idxs])
    print(f"  {name:<26} {snr:>7.2f}  {react:>7.2f}x {corr:>8.4f}  {vep:>8.1f}")

## 14. Key Findings & Discussion

### Recording Setup (Confirmed)
1. **1000 Hz, 0.1 µV/bit**, hardware bandpass 0.1–250 Hz, notch OFF during recording
2. **Reference:** mastoid (behind ear), **Ground:** base of skull — both dedicated amplifier inputs, not data channels
3. **60 S7 triggers** in 3 blocks + **7 eyes-open/close markers** → full epoch segmentation
4. **Pre-averaged VEP** from Recorder: 74 segments, 0.5–30 Hz + 60 Hz notch

### Alpha Detection
5. **Berger effect** (alpha reactivity ratio > 1 during eyes-closed) — check Section 9 results for each electrode type
6. **Alpha envelope dynamics** — check Section 11 for whether tEEG tracks the same temporal pattern as disc
7. **Correlation with disc** — Section 12 quantifies how well each channel tracks Ch11

### tEEG vs Conventional
8. Compare **alpha SNR, reactivity, disc correlation, and VEP amplitude** in Section 13 summary dashboard
9. tEEG channels have lower absolute amplitude (expected from Laplacian spatial filtering) but potentially better SNR
10. **Clipping** in conventional channels (Ch1,3,5,7) at ±3276.7 µV may affect power estimates

### Caveats
- **Channel mapping** (odd=Conv, even=tEEG) is inferred from signal amplitude — **verify with professor**
- Electrode scalp positions not documented in these files
- Only 3 eyes-closed epochs (limited stats)
- No artifact rejection (consider ICA for blinks/muscle)
- The last epoch (Eyes Open #4 at 342.5s) has no closing marker — may extend to end of recording
